<a href="https://colab.research.google.com/github/caffeicsatyam/radiology-reporting-harness/blob/main/Radiology_reporting_harness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RADIOLOGY REPORT HARNESS

In [1]:
!pip install -U --no-cache-dir \
    "trl>=0.28.0" \
    "transformers>=4.57.0" \
    "peft>=0.17.0" \
    "accelerate>=1.10.0" \
    "datasets>=3.6.0" \
    "bitsandbytes>=0.46.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 390.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 331.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 307.0 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [2]:
import os
import pandas as pd
import numpy as np
import json
import warnings
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from peft import PeftModel, LoraConfig
from trl import SFTTrainer
warnings.filterwarnings('ignore')

In [3]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)

# MODEL TOKENIZER

In [4]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-3B-Instruct", device_map="auto")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

# BNB CONFIG

In [5]:
# ============================================================
# BITSANDBYTES / 4-BIT QLoRA
# ============================================================

use_4bit = True

# NOTE: compute dtype is fixed to float16 below (not GPU-detected) so it
# always matches the fp16 casting done later in this notebook.

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# TRAINING ARGUEMENT

In [6]:
# ============================================================
# TRAINING ARGUMENTS
# ============================================================

output_dir = "/kaggle/working/qwen2.5-3b-radiology"

num_train_epochs = 3

# T4 -> FP16
fp16 = not torch.cuda.is_bf16_supported()
bf16 = torch.cuda.is_bf16_supported()

# Batch size
per_device_train_batch_size = 1
per_device_eval_batch_size = 1

# Effective batch size = 1 x 8 = 8
gradient_accumulation_steps = 8

# 8-bit paged optimizer
optim = "paged_adamw_8bit"

# Save VRAM
gradient_checkpointing = True

# Gradient clipping
max_grad_norm = 1.0

# QLoRA learning rate
learning_rate = 2e-4

# Small dataset -> mild regularization
weight_decay = 0.01

# Stable for initial experiment
lr_scheduler_type = "constant"

max_steps = -1

# 3% warmup
warmup_ratio = 0.03

# Group similar-length samples
group_by_length = True

# Checkpoints
save_strategy = "steps"
save_steps = 50
save_total_limit = 2

# Logging
logging_steps = 10

# Evaluation
eval_strategy = "steps"
eval_steps = 50

# Load best checkpoint after training
load_best_model_at_end = True

# Best model metric
metric_for_best_model = "eval_loss"
greater_is_better = False
report_to = "none"


# ============================================================
# SFT PARAMETERS
# ============================================================

max_seq_length = 2048
packing = False


# LOADING TOKENIZER AND MODEL

In [7]:
# ============================================================
# TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

# Qwen uses EOS as padding if PAD isn't defined
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("\nTokenizer:")
print("PAD:", tokenizer.pad_token)
print("EOS:", tokenizer.eos_token)
print("BOS:", tokenizer.bos_token)


Tokenizer:
PAD: <|endoftext|>
EOS: <|im_end|>
BOS: None


# LOAD MODEL

In [8]:
# ============================================================
# LOAD MODEL
# ============================================================
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
    trust_remote_code=True
)


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [9]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(
    model
)

for param in model.parameters():

    if param.dtype == torch.bfloat16:
        param.data = param.data.to(
            torch.float16
        )


# LOADING DATASET

In [11]:
# ============================================================
# PATHS
# ============================================================

TRAIN_JSONL = "/content/train_sft_train.jsonl"
VAL_JSONL = "/content/train_sft_val.jsonl"


# ============================================================
# LOAD DATASETS
# ============================================================

train_dataset = load_dataset(
    "json",
    data_files=TRAIN_JSONL,
    split="train"
)

val_dataset = load_dataset(
    "json",
    data_files=VAL_JSONL,
    split="train"
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Train: 540
Validation: 96


# FORMATTING

In [12]:
# ============================================================
# FORMAT ONE EXAMPLE
# ============================================================

def format_example(example):

    # --------------------------------------------------------
    # Nested input
    # --------------------------------------------------------

    input_data = example["input"]

    template_content = str(
        input_data.get("template_content", "")
    )

    dictation = str(
        input_data.get("dictation", "")
    )

    modality = str(
        input_data.get("modality", "")
    )

    body_part = str(
        input_data.get("body_part", "")
    )

    study_description = str(
        input_data.get("study_description", "")
    )

    patient_age_band = str(
        input_data.get("patient_age_band", "")
    )

    patient_sex = str(
        input_data.get("patient_sex", "")
    )

    # --------------------------------------------------------
    # Target
    # --------------------------------------------------------

    output_data = example["output"]

    target_report = str(
        output_data.get("report", "")
    )

    # --------------------------------------------------------
    # User message
    # --------------------------------------------------------

    user_prompt = f"""
Generate the final radiology report from the supplied template and dictation.

Requirements:
- Incorporate all relevant dictated findings accurately.
- Preserve the template's structure and wording where appropriate.
- Do not invent findings.
- Produce exactly two top-level sections:
  FINDINGS:
  IMPRESSION:
- Return only the final report.

MODALITY:
{modality}

BODY PART:
{body_part}

STUDY DESCRIPTION:
{study_description}

PATIENT AGE BAND:
{patient_age_band}

PATIENT SEX:
{patient_sex}

TEMPLATE:
{template_content}

DICTATION:
{dictation}
""".strip()

    # --------------------------------------------------------
    # Qwen messages
    # --------------------------------------------------------

    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert radiology report editor. "
                "Generate the final report from the supplied "
                "template and dictation. Preserve template "
                "structure and wording where appropriate. "
                "Do not invent findings."
            )
        },
        {
            "role": "user",
            "content": user_prompt
        },
        {
            "role": "assistant",
            "content": target_report
        }
    ]

    # --------------------------------------------------------
    # Apply Qwen chat template
    # --------------------------------------------------------

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {
        "text": text
    }


# ============================================================
# TEST FORMAT BEFORE MAP
# ============================================================

print("\n" + "=" * 80)
print("FORMAT TEST")
print("=" * 80)

formatted_test = format_example(
    train_dataset[0]
)

print(formatted_test["text"])


# ============================================================
# FORMAT DATASETS
# ============================================================

train_dataset = train_dataset.map(
    format_example,
    remove_columns=train_dataset.column_names,
    desc="Formatting training dataset"
)

val_dataset = val_dataset.map(
    format_example,
    remove_columns=val_dataset.column_names,
    desc="Formatting validation dataset"
)


# ============================================================
# VERIFY FORMATTING
# ============================================================

print("\n" + "=" * 80)
print("FORMATTED DATASET")
print("=" * 80)

print("Train columns:", train_dataset.column_names)
print("Validation columns:", val_dataset.column_names)

print("\nFormatted example:")
print(train_dataset[0]["text"])





FORMAT TEST
<|im_start|>system
You are an expert radiology report editor. Generate the final report from the supplied template and dictation. Preserve template structure and wording where appropriate. Do not invent findings.<|im_end|>
<|im_start|>user
Generate the final radiology report from the supplied template and dictation.

Requirements:
- Incorporate all relevant dictated findings accurately.
- Preserve the template's structure and wording where appropriate.
- Do not invent findings.
- Produce exactly two top-level sections:
  FINDINGS:
  IMPRESSION:
- Return only the final report.

MODALITY:
MRI

BODY PART:
Leg

STUDY DESCRIPTION:
MRI TIB/FIB LEFT

PATIENT AGE BAND:
65-69

PATIENT SEX:
female

TEMPLATE:
FINDINGS:
OSSEOUS STRUCTURES: No acute fracture or marrow abnormality of the tibial or fibular shafts. No aggressive bone lesion.
MUSCULATURE: Normal compartmental signal. No edema or mass.
SOFT TISSUES: No fluid collection or soft tissue mass.
OTHER FINDINGS:


IMPRESSION:
No a

Formatting training dataset:   0%|          | 0/540 [00:00<?, ? examples/s]

Formatting validation dataset:   0%|          | 0/96 [00:00<?, ? examples/s]


FORMATTED DATASET
Train columns: ['text']
Validation columns: ['text']

Formatted example:
<|im_start|>system
You are an expert radiology report editor. Generate the final report from the supplied template and dictation. Preserve template structure and wording where appropriate. Do not invent findings.<|im_end|>
<|im_start|>user
Generate the final radiology report from the supplied template and dictation.

Requirements:
- Incorporate all relevant dictated findings accurately.
- Preserve the template's structure and wording where appropriate.
- Do not invent findings.
- Produce exactly two top-level sections:
  FINDINGS:
  IMPRESSION:
- Return only the final report.

MODALITY:
MRI

BODY PART:
Leg

STUDY DESCRIPTION:
MRI TIB/FIB LEFT

PATIENT AGE BAND:
65-69

PATIENT SEX:
female

TEMPLATE:
FINDINGS:
OSSEOUS STRUCTURES: No acute fracture or marrow abnormality of the tibial or fibular shafts. No aggressive bone lesion.
MUSCULATURE: Normal compartmental signal. No edema or mass.
SOFT TISSU

In [13]:
# ============================================================
# TOKEN LENGTH CHECK
# ============================================================

print("\n" + "=" * 80)
print("TOKEN LENGTH CHECK")
print("=" * 80)

token_lengths = []

for example in train_dataset:

    encoded = tokenizer(
        example["text"],
        add_special_tokens=False,
        truncation=False
    )

    token_lengths.append(
        len(encoded["input_ids"])
    )

print("Minimum:", min(token_lengths))
print("Maximum:", max(token_lengths))
print(
    "Average:",
    sum(token_lengths) / len(token_lengths)
)

sorted_lengths = sorted(token_lengths)

for percentile in [50, 90, 95, 99]:

    idx = min(
        int(len(sorted_lengths) * percentile / 100),
        len(sorted_lengths) - 1
    )

    print(
        f"{percentile}th percentile:",
        sorted_lengths[idx]
    )


TOKEN LENGTH CHECK
Minimum: 275
Maximum: 2106
Average: 687.6722222222222
50th percentile: 572
90th percentile: 1276
95th percentile: 1413
99th percentile: 1654


# CONFIG

In [14]:
from trl import SFTConfig

# ============================================================
# PEFT / QLORA CONFIG
# ============================================================

peft_config = LoraConfig(

    r=16,

    lora_alpha=32,

    lora_dropout=0.05,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],

    bias="none",

    task_type="CAUSAL_LM"
)


# ============================================================
# SFT CONFIG
# ============================================================

sft_config = SFTConfig(

    output_dir=output_dir,

    # Training
    num_train_epochs=3,

    per_device_train_batch_size=1,

    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    weight_decay=0.01,

    max_grad_norm=1.0,

    max_steps=-1,

    # Precision
    # NOTE: Trainer-side fp16/bf16 mixed precision is disabled on purpose.
    # The base model already computes in float16 via bnb_4bit_compute_dtype,
    # and enabling fp16 here creates a GradScaler that crashes
    # ("_amp_foreach_non_finite_check_and_unscale_cuda not implemented for
    # 'BFloat16'") because the LoRA adapter layers PEFT injects are created
    # in bfloat16, not float16. Leaving both off avoids the GradScaler
    # entirely; the LoRA adapters (a small fraction of total params) simply
    # train with fp32 gradients.
    fp16=False,

    bf16=False,

    # Optimizer
    optim="paged_adamw_8bit",

    lr_scheduler_type="constant",

    warmup_steps=10,

    # Memory
    gradient_checkpointing=True,

    use_cache=False,

    # Sequence
    max_length=2048,

    packing=False,

    # Evaluation
    eval_strategy="steps",

    eval_steps=50,

    # Saving
    save_strategy="steps",

    save_steps=50,

    save_total_limit=2,

    # Logging
    logging_strategy="steps",

    logging_steps=10,

    report_to="none",

    # Best checkpoint
    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",

    greater_is_better=False,

    # Dataset
    dataset_text_field="text",

    seed=42
)

# TRAINER

In [15]:
# ============================================================
# CREATE SFT TRAINER
# ============================================================

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    peft_config=peft_config
)


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 80)
print("SFT TRAINER READY")
print("=" * 80)

print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print("Model:", MODEL_ID)
print("LoRA rank:", peft_config.r)
print("Max length:", 2048)

Adding EOS to train dataset:   0%|          | 0/540 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/540 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/540 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/540 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/540 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/96 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/96 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/96 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/96 [00:00<?, ? examples/s]


SFT TRAINER READY
Training examples: 540
Validation examples: 96
Model: Qwen/Qwen2.5-3B-Instruct
LoRA rank: 16
Max length: 2048


In [16]:
# ============================================================
# CHECK TOKEN LENGTHS
# ============================================================

lengths = []

for example in train_dataset:

    tokens = tokenizer(
        example["text"],
        add_special_tokens=True,
        truncation=False
    )

    lengths.append(
        len(tokens["input_ids"])
    )

print("Maximum tokens:", max(lengths))
print("Average tokens:", sum(lengths) / len(lengths))

for p in [50, 90, 95, 99]:
    print(
        f"{p}th percentile:",
        sorted(lengths)[int(len(lengths) * p / 100)]
    )

Maximum tokens: 2106
Average tokens: 687.6722222222222
50th percentile: 572
90th percentile: 1276
95th percentile: 1413
99th percentile: 1654


In [17]:
# ============================================================
# VERIFY ACTUAL TRAINING TEXT
# ============================================================

print("Number of training examples:", len(train_dataset))

for i in range(3):
    text = train_dataset[i]["text"]

    print("\n" + "=" * 80)
    print("EXAMPLE", i)
    print("Character count:", len(text))
    print("Word count:", len(text.split()))
    print("First 1000 characters:\n")
    print(text[:1000])


# ============================================================
# TOKEN COUNT WITHOUT TRUNCATION
# ============================================================

token_lengths = []

for i in range(len(train_dataset)):

    text = train_dataset[i]["text"]

    encoded = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False
    )

    token_lengths.append(
        len(encoded["input_ids"])
    )


print("\n" + "=" * 80)
print("TOKEN LENGTHS")
print("=" * 80)

print("Minimum:", min(token_lengths))
print("Maximum:", max(token_lengths))
print("Average:", sum(token_lengths) / len(token_lengths))

sorted_lengths = sorted(token_lengths)

for percentile in [50, 90, 95, 99]:

    idx = min(
        int(len(sorted_lengths) * percentile / 100),
        len(sorted_lengths) - 1
    )

    print(
        f"{percentile}th percentile:",
        sorted_lengths[idx]
    )

Number of training examples: 540

EXAMPLE 0
Character count: 3313
Word count: 439
First 1000 characters:

<|im_start|>system
You are an expert radiology report editor. Generate the final report from the supplied template and dictation. Preserve template structure and wording where appropriate. Do not invent findings.<|im_end|>
<|im_start|>user
Generate the final radiology report from the supplied template and dictation.

Requirements:
- Incorporate all relevant dictated findings accurately.
- Preserve the template's structure and wording where appropriate.
- Do not invent findings.
- Produce exactly two top-level sections:
  FINDINGS:
  IMPRESSION:
- Return only the final report.

MODALITY:
MRI

BODY PART:
Leg

STUDY DESCRIPTION:
MRI TIB/FIB LEFT

PATIENT AGE BAND:
65-69

PATIENT SEX:
female

TEMPLATE:
FINDINGS:
OSSEOUS STRUCTURES: No acute fracture or marrow abnormality of the tibial or fibular shafts. No aggressive bone lesion.
MUSCULATURE: Normal compartmental signal. No edema or ma

# TRAINING

In [18]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,0.498085,0.410935,0.400164,271516.000000,0.899551
100,0.364068,0.350110,0.345368,546984.000000,0.912255
150,0.258822,0.337235,0.280229,818315.000000,0.917218
200,0.278417,0.326106,0.271943,1094251.000000,0.919235
204,0.278417,0.323069,0.268359,1115472.000000,0.920414


TrainOutput(global_step=204, training_loss=0.43219853674664216, metrics={'train_runtime': 4281.6586, 'train_samples_per_second': 0.378, 'train_steps_per_second': 0.048, 'total_flos': 1.8771434955472896e+16, 'train_loss': 0.43219853674664216, 'epoch': 3.0})

In [23]:
# ============================================================
# SAVE TRAINED QLORA ADAPTER — GOOGLE COLAB
# ============================================================

SAVE_DIR = "/content/qwen2.5-3b-radiology-lora"

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Saved trained adapter to:")
print(SAVE_DIR)

Saved trained adapter to:
/content/qwen2.5-3b-radiology-lora


In [25]:
from tqdm.auto import tqdm

In [20]:
|# ============================================================
# VALIDATION: GENERATE REPORTS + LOCAL RES
# Google Colab version
# ============================================================

import os
import re
import json
import unicodedata
import pandas as pd
import numpy as np
import torch

from datasets import load_dataset


# ============================================================
# CONFIG
# ============================================================

VAL_JSONL = "/content/train_sft_val.jsonl"

PREDICTIONS_CSV = "/content/validation_predictions.csv"
SCORES_CSV = "/content/validation_scores.csv"

MAX_INPUT_LENGTH = 2048
MAX_NEW_TOKENS = 1024


# ============================================================
# CHECK FILES
# ============================================================

if not os.path.exists(VAL_JSONL):
    raise FileNotFoundError(
        f"Validation file not found: {VAL_JSONL}"
    )

print("Validation file:", VAL_JSONL)


# ============================================================
# CHECK TRAINER / TOKENIZER / MODEL
# ============================================================

if "trainer" not in globals():
    raise RuntimeError(
        "trainer not found. Make sure training completed."
    )

if "tokenizer" not in globals():
    raise RuntimeError(
        "tokenizer not found."
    )

model = trainer.model
model.eval()

print("Model:", MODEL_ID)


# ============================================================
# LOAD RAW VALIDATION DATA
# ============================================================

val_raw = load_dataset(
    "json",
    data_files=VAL_JSONL,
    split="train"
)

print("Validation examples:", len(val_raw))


# ============================================================
# BUILD QWEN PROMPT
# ============================================================

def build_messages(record):

    x = record["input"]

    return [
        {
            "role": "system",
            "content": (
                "You are an expert radiology report editor. "
                "Generate the final report from the supplied "
                "template and dictation. Preserve template "
                "structure and wording where appropriate. "
                "Do not invent findings."
            )
        },
        {
            "role": "user",
            "content": f"""
Generate the final radiology report from the supplied template and dictation.

Requirements:
- Incorporate all relevant dictated findings accurately.
- Preserve the template's structure and wording where appropriate.
- Do not invent findings.
- Produce exactly two top-level sections:
  FINDINGS:
  IMPRESSION:
- Return only the final report.

MODALITY:
{x.get("modality", "")}

BODY PART:
{x.get("body_part", "")}

STUDY DESCRIPTION:
{x.get("study_description", "")}

PATIENT AGE BAND:
{x.get("patient_age_band", "")}

PATIENT SEX:
{x.get("patient_sex", "")}

TEMPLATE:
{x.get("template_content", "")}

DICTATION:
{x.get("dictation", "")}
""".strip()
        }
    ]


# ============================================================
# GENERATE REPORT
# ============================================================

@torch.no_grad()
def generate_report(record):

    messages = build_messages(record)

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LENGTH
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True
    )

    input_len = inputs["input_ids"].shape[1]

    generated_tokens = outputs[
        0,
        input_len:
    ]

    text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return text


# ============================================================
# CLEAN GENERATED REPORT
# ============================================================

def clean_generated_report(text):

    text = str(text).strip()

    # Remove chat artifacts
    text = text.replace(
        "<|im_end|>",
        ""
    ).strip()

    # Find FINDINGS
    match = re.search(
        r"\bFINDINGS\s*:",
        text,
        flags=re.IGNORECASE
    )

    if match:
        text = text[match.start():]

    # Normalize headers
    text = re.sub(
        r"^\s*FINDINGS\s*:",
        "FINDINGS:",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\n\s*IMPRESSION\s*:",
        "\n\nIMPRESSION:",
        text,
        flags=re.IGNORECASE
    )

    return text.strip()


# ============================================================
# GENERATE ALL VALIDATION PREDICTIONS
# ============================================================

predictions = []

print("\n" + "=" * 80)
print("GENERATING VALIDATION PREDICTIONS")
print("=" * 80)

for i, record in enumerate(val_raw):

    prediction = generate_report(
        record
    )

    prediction = clean_generated_report(
        prediction
    )

    predictions.append(
        prediction
    )

    if (i + 1) % 10 == 0:
        print(
            f"{i + 1}/{len(val_raw)} completed"
        )


# ============================================================
# SAVE PREDICTIONS
# ============================================================

prediction_rows = []

for i, record in enumerate(val_raw):

    x = record["input"]

    prediction_rows.append({

        "case_id":
            x.get("case_id", i),

        "modality":
            x.get("modality", ""),

        "body_part":
            x.get("body_part", ""),

        "template_content":
            x.get("template_content", ""),

        "dictation":
            x.get("dictation", ""),

        "reference_report":
            record["output"]["report"],

        "predicted_report":
            predictions[i]
    })


pred_df = pd.DataFrame(
    prediction_rows
)

pred_df.to_csv(
    PREDICTIONS_CSV,
    index=False,
    encoding="utf-8"
)

print(
    "\nSaved predictions:",
    PREDICTIONS_CSV
)


# ============================================================
# SCORING NORMALIZATION
# ============================================================

UNITS = {
    "millimeters": "mm",
    "millimeter": "mm",
    "centimeters": "cm",
    "centimeter": "cm"
}

FUNCTION_WORDS = {
    "a", "an", "the", "and", "or",
    "of", "to", "in", "on", "at",
    "for", "with", "without",
    "from", "by", "as", "is",
    "are", "was", "were", "be",
    "been", "being", "this",
    "that", "these", "those",
    "there", "it", "its",
    "than", "into", "over",
    "under", "within"
}

CRITICAL_WORDS = {
    "no", "not", "without",
    "negative", "absent",
    "right", "left", "bilateral",
    "mild", "moderate", "severe",
    "marked", "minimal", "trace",
    "small", "large",
    "acute", "chronic", "subacute",
    "new", "stable",
    "fracture", "dislocation",
    "tear", "rupture",
    "effusion", "pneumothorax",
    "pneumonia", "mass", "lesion",
    "stenosis", "occlusion",
    "thrombosis", "edema",
    "swelling"
}

UNIT_WORDS = {
    "mm",
    "cm"
}


def normalize_for_score(text):

    if text is None:
        return ""

    text = str(text)

    text = unicodedata.normalize(
        "NFKC",
        text
    )

    text = text.lower()

    # Units
    for source, target in UNITS.items():

        text = re.sub(
            rf"\b{re.escape(source)}\b",
            target,
            text
        )

    # List markers
    text = re.sub(
        r"(?m)^\s*(?:\d+[\.\)]|[-*•])\s+",
        "",
        text
    )

    # air-space -> airspace
    text = re.sub(
        r"(?<=[a-z])-(?=[a-z])",
        "",
        text
    )

    # Letter-number boundaries
    text = re.sub(
        r"(?<=[A-Za-z])(?=\d)",
        " ",
        text
    )

    text = re.sub(
        r"(?<=\d)(?=[A-Za-z])",
        " ",
        text
    )

    # Preserve signed numbers
    protected = []

    def protect(match):

        value = match.group(0)

        token = f"__SIGNED_{len(protected)}__"

        protected.append(value)

        return token

    text = re.sub(
        r"[+-]\d+(?:\.\d+)?",
        protect,
        text
    )

    # Remove punctuation
    text = re.sub(
        r"[^a-z0-9_\s]",
        " ",
        text
    )

    # Restore signs
    for i, value in enumerate(protected):

        text = text.replace(
            f"__signed_{i}__".lower(),
            value
        )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


def tokenize_score(text):

    text = normalize_for_score(
        text
    )

    if not text:
        return []

    return text.split()


# ============================================================
# TOKEN WEIGHT
# ============================================================

def token_weight(token):

    if re.fullmatch(
        r"[+-]?\d+(?:\.\d+)?",
        token
    ):
        return 4.0

    if token in UNIT_WORDS:
        return 4.0

    if token in CRITICAL_WORDS:
        return 4.0

    if token in FUNCTION_WORDS:
        return 0.25

    return 2.0


# ============================================================
# WEIGHTED LEVENSHTEIN
# ============================================================

def weighted_levenshtein(
    reference_tokens,
    prediction_tokens
):

    n = len(reference_tokens)
    m = len(prediction_tokens)

    if n == 0 and m == 0:
        return 0.0

    dp = np.zeros(
        (n + 1, m + 1),
        dtype=np.float64
    )

    for i in range(1, n + 1):

        dp[i, 0] = (
            dp[i - 1, 0]
            +
            token_weight(
                reference_tokens[i - 1]
            )
        )

    for j in range(1, m + 1):

        dp[0, j] = (
            dp[0, j - 1]
            +
            token_weight(
                prediction_tokens[j - 1]
            )
        )

    for i in range(1, n + 1):

        ref = reference_tokens[i - 1]
        ref_weight = token_weight(ref)

        for j in range(1, m + 1):

            pred = prediction_tokens[j - 1]
            pred_weight = token_weight(pred)

            deletion = (
                dp[i - 1, j]
                +
                ref_weight
            )

            insertion = (
                dp[i, j - 1]
                +
                pred_weight
            )

            substitution = (
                dp[i - 1, j - 1]
                +
                (
                    0.0
                    if ref == pred
                    else max(
                        ref_weight,
                        pred_weight
                    )
                )
            )

            dp[i, j] = min(
                deletion,
                insertion,
                substitution
            )

    ref_weight_total = sum(
        token_weight(x)
        for x in reference_tokens
    )

    pred_weight_total = sum(
        token_weight(x)
        for x in prediction_tokens
    )

    denominator = max(
        ref_weight_total,
        pred_weight_total
    )

    if denominator == 0:
        return 0.0

    return min(
        dp[n, m] / denominator,
        1.0
    )


def word_edit_score(
    reference,
    prediction
):

    return weighted_levenshtein(
        tokenize_score(reference),
        tokenize_score(prediction)
    )


# ============================================================
# EXTRACT REPORT SECTIONS
# ============================================================

def extract_report_sections(report):

    report = str(report)

    findings_match = re.search(
        r"FINDINGS\s*:\s*(.*?)(?=\n\s*IMPRESSION\s*:|\Z)",
        report,
        flags=re.IGNORECASE | re.DOTALL
    )

    impression_match = re.search(
        r"IMPRESSION\s*:\s*(.*)\Z",
        report,
        flags=re.IGNORECASE | re.DOTALL
    )

    findings = (
        findings_match.group(1).strip()
        if findings_match
        else ""
    )

    impression = (
        impression_match.group(1).strip()
        if impression_match
        else ""
    )

    return findings, impression


# ============================================================
# PARSE LABELED FIELDS
# ============================================================

def parse_labeled_fields(text):

    fields = {}

    if not text:
        return fields

    lines = text.splitlines()

    current_label = None
    current_text = []

    pattern = re.compile(
        r"^\s*([A-Z][A-Z0-9 /&(),.\-]*?)\s*:\s*(.*)$"
    )

    for line in lines:

        match = pattern.match(line)

        if match:

            if current_label is not None:

                content = " ".join(
                    current_text
                ).strip()

                if content:
                    fields[current_label] = content

            current_label = match.group(1).strip()

            current_text = []

            if match.group(2).strip():

                current_text.append(
                    match.group(2).strip()
                )

        else:

            if current_label is not None:
                if line.strip():
                    current_text.append(
                        line.strip()
                    )

    if current_label is not None:

        content = " ".join(
            current_text
        ).strip()

        if content:
            fields[current_label] = content

    return fields


# ============================================================
# FINDINGS SCORE
# ============================================================

def calculate_findings_score(
    template,
    reference,
    prediction
):

    reference_findings, _ = (
        extract_report_sections(
            reference
        )
    )

    prediction_findings, _ = (
        extract_report_sections(
            prediction
        )
    )

    template_findings, _ = (
        extract_report_sections(
            template
        )
    )

    reference_fields = parse_labeled_fields(
        reference_findings
    )

    prediction_fields = parse_labeled_fields(
        prediction_findings
    )

    template_fields = parse_labeled_fields(
        template_findings
    )

    # --------------------------------------------------------
    # If no useful labels are present, compare whole FINDINGS
    # --------------------------------------------------------

    if not reference_fields:

        return word_edit_score(
            reference_findings,
            prediction_findings
        )

    # --------------------------------------------------------
    # Field-aware calculation
    # --------------------------------------------------------

    numerator = 0.0
    denominator = 0.0

    # Fields in reference
    for field, reference_text in (
        reference_fields.items()
    ):

        template_text = (
            template_fields.get(
                field,
                ""
            )
        )

        prediction_text = (
            prediction_fields.get(
                field,
                ""
            )
        )

        changed = (
            normalize_for_score(
                template_text
            )
            !=
            normalize_for_score(
                reference_text
            )
        )

        field_weight = (
            3.0
            if changed
            else 1.0
        )

        field_score = word_edit_score(
            reference_text,
            prediction_text
        )

        numerator += (
            field_weight
            *
            field_score
        )

        denominator += field_weight

    # Missing expected reference fields
    for field in template_fields:

        if field not in reference_fields:

            reference_text = ""

            prediction_text = (
                prediction_fields.get(
                    field,
                    ""
                )
            )

            field_score = word_edit_score(
                reference_text,
                prediction_text
            )

            numerator += field_score
            denominator += 1.0

    if denominator == 0:
        return word_edit_score(
            reference_findings,
            prediction_findings
        )

    return min(
        numerator / denominator,
        1.0
    )


# ============================================================
# IMPRESSION SCORE
# ============================================================

def calculate_impression_score(
    reference,
    prediction
):

    _, reference_impression = (
        extract_report_sections(
            reference
        )
    )

    _, prediction_impression = (
        extract_report_sections(
            prediction
        )
    )

    return word_edit_score(
        reference_impression,
        prediction_impression
    )


# ============================================================
# CALCULATE FINAL RES
# ============================================================

results = []

for i, record in enumerate(val_raw):

    template = str(
        record["input"].get(
            "template_content",
            ""
        )
    )

    reference = str(
        record["output"].get(
            "report",
            ""
        )
    )

    prediction = predictions[i]

    f_score = calculate_findings_score(
        template,
        reference,
        prediction
    )

    i_score = calculate_impression_score(
        reference,
        prediction
    )

    res = (
        0.65 * f_score
        +
        0.35 * i_score
    )

    results.append({

        "case_id":
            record["input"].get(
                "case_id",
                i
            ),

        "findings_score":
            f_score,

        "impression_score":
            i_score,

        "res":
            res
    })


results_df = pd.DataFrame(
    results
)


# ============================================================
# RESULTS
# ============================================================

mean_findings = (
    results_df["findings_score"].mean()
)

mean_impression = (
    results_df["impression_score"].mean()
)

mean_res = (
    results_df["res"].mean()
)

exact_matches = (
    results_df["res"] == 0
).sum()


print("\n" + "=" * 80)
print("VALIDATION RESULTS")
print("=" * 80)

print(
    f"Cases: {len(results_df)}"
)

print(
    f"Mean FINDINGS score:   {mean_findings:.6f}"
)

print(
    f"Mean IMPRESSION score: {mean_impression:.6f}"
)

print(
    f"Mean RES:              {mean_res:.6f}"
)

print(
    f"Exact matches:         {exact_matches}/{len(results_df)}"
)


# ============================================================
# SAVE SCORES
# ============================================================

# Merge scores into prediction dataframe
pred_df = pred_df.merge(
    results_df,
    on="case_id",
    how="left"
)

pred_df.to_csv(
    PREDICTIONS_CSV,
    index=False,
    encoding="utf-8"
)

results_df.to_csv(
    SCORES_CSV,
    index=False,
    encoding="utf-8"
)

print(
    "\nSaved:",
    PREDICTIONS_CSV
)

print(
    "Saved:",
    SCORES_CSV
)


# ============================================================
# WORST CASES
# ============================================================

print("\n" + "=" * 80)
print("WORST 10 CASES")
print("=" * 80)

print(
    pred_df.sort_values(
        "res",
        ascending=False
    )[
        [
            "case_id",
            "findings_score",
            "impression_score",
            "res"
        ]
    ].head(10).to_string(
        index=False
    )
)

Validation file: /content/train_sft_val.jsonl
Model: Qwen/Qwen2.5-3B-Instruct
Validation examples: 96

GENERATING VALIDATION PREDICTIONS
10/96 completed


KeyboardInterrupt: 

In [26]:
# ============================================================
# PATHS
# ============================================================

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# Change this ONLY if you saved the adapter somewhere else
ADAPTER_PATH = "/content/qwen2.5-3b-radiology-lora"

# Your test CSV
TEST_CSV = "/content/test.csv"

# Submission output
SUBMISSION_CSV = "/content/submission.csv"


# ============================================================
# LOAD TEST DATA
# ============================================================

test_df = pd.read_csv(TEST_CSV)

print("Test shape:", test_df.shape)
print("Columns:", test_df.columns.tolist())


# ============================================================
# LOAD TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"


# ============================================================
# 4-BIT CONFIG
# ============================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)


# ============================================================
# LOAD BASE MODEL
# ============================================================

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

base_model.config.use_cache = True


# ============================================================
# LOAD TRAINED LORA ADAPTER
# ============================================================

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

model.eval()

print("Model + LoRA adapter loaded.")


# ============================================================
# BUILD PROMPT
# ============================================================

def build_messages(row):

    def get(col):
        if col not in row.index:
            return ""
        value = row[col]
        if pd.isna(value):
            return ""
        return str(value)

    return [
        {
            "role": "system",
            "content": (
                "You are an expert radiology report editor. "
                "Generate the final report from the supplied "
                "template and dictation. Preserve the template "
                "structure and wording where appropriate. "
                "Do not invent findings."
            )
        },
        {
            "role": "user",
            "content": f"""
Generate the final radiology report from the supplied template and dictation.

Requirements:
- Incorporate all relevant dictated findings accurately.
- Preserve the template's structure and wording where appropriate.
- Do not invent findings.
- Produce exactly two top-level sections:
  FINDINGS:
  IMPRESSION:
- Return only the final report.

MODALITY:
{get("modality")}

BODY PART:
{get("body_part")}

STUDY DESCRIPTION:
{get("study_description")}

PATIENT AGE BAND:
{get("patient_age_band")}

PATIENT SEX:
{get("patient_sex")}

TEMPLATE:
{get("template_content")}

DICTATION:
{get("dictation")}
""".strip()
        }
    ]


# ============================================================
# GENERATE REPORT
# ============================================================

@torch.no_grad()
def generate_report(row):

    messages = build_messages(row)

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,
        do_sample=False,
        num_beams=1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True
    )

    input_len = inputs["input_ids"].shape[1]

    generated_tokens = outputs[
        0,
        input_len:
    ]

    text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return text


# ============================================================
# CLEAN REPORT
# ============================================================

def clean_report(text):

    text = str(text).strip()

    # Remove accidental chat artifacts
    text = re.sub(
        r"<\|im_end\|>",
        "",
        text
    ).strip()

    # Start from FINDINGS
    match = re.search(
        r"\bFINDINGS\s*:",
        text,
        flags=re.IGNORECASE
    )

    if match:
        text = text[match.start():]

    # Normalize headers
    text = re.sub(
        r"^\s*FINDINGS\s*:",
        "FINDINGS:",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\n\s*IMPRESSION\s*:",
        "\n\nIMPRESSION:",
        text,
        flags=re.IGNORECASE
    )

    # --------------------------------------------------------
    # Ensure only the required top-level structure.
    # --------------------------------------------------------

    if "IMPRESSION:" not in text:

        text = text.rstrip()

        text += (
            "\n\nIMPRESSION:\n"
            "No acute abnormality."
        )

    return text.strip()


# ============================================================
# GENERATE ALL TEST REPORTS
# ============================================================

predictions = []

print("\n" + "=" * 80)
print("GENERATING TEST PREDICTIONS")
print("=" * 80)

for _, row in tqdm(
    test_df.iterrows(),
    total=len(test_df),
    desc="Generating reports"
):

    prediction = generate_report(row)

    prediction = clean_report(
        prediction
    )

    predictions.append(
        prediction
    )


# ============================================================
# BUILD SUBMISSION
# ============================================================

submission = pd.DataFrame({
    "case_id": test_df["case_id"],
    "report": predictions
})


# ============================================================
# FINAL VALIDATION
# ============================================================

assert len(submission) == len(test_df)

assert submission["case_id"].nunique() == len(
    submission
), "Duplicate case_id found."

assert set(submission["case_id"]) == set(
    test_df["case_id"]
), "case_id mismatch."

assert submission["report"].notna().all()

assert submission["report"].str.strip().ne("").all()


# Check required sections
bad_structure = submission[
    ~submission["report"].str.contains(
        r"FINDINGS\s*:",
        case=False,
        regex=True
    )
    |
    ~submission["report"].str.contains(
        r"IMPRESSION\s*:",
        case=False,
        regex=True
    )
]

if len(bad_structure) > 0:

    print(
        f"WARNING: {len(bad_structure)} reports have "
        "missing FINDINGS or IMPRESSION."
    )


# ============================================================
# SAVE
# ============================================================

submission.to_csv(
    SUBMISSION_CSV,
    index=False,
    quoting=1,   # csv.QUOTE_ALL
    encoding="utf-8"
)


# ============================================================
# VERIFY CSV
# ============================================================

check = pd.read_csv(
    SUBMISSION_CSV
)

print("\n" + "=" * 80)
print("SUBMISSION READY")
print("=" * 80)

print("Rows:", len(check))
print("Columns:", check.columns.tolist())
print("File:", SUBMISSION_CSV)

print("\nFirst prediction:")
print(check.iloc[0]["report"])

print("\nSaved successfully.")

Test shape: (132, 8)
Columns: ['case_id', 'modality', 'body_part', 'study_description', 'patient_age_band', 'patient_sex', 'template_content', 'dictation']


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model + LoRA adapter loaded.

GENERATING TEST PREDICTIONS


Generating reports:   0%|          | 0/132 [00:00<?, ?it/s]


SUBMISSION READY
Rows: 132
Columns: ['case_id', 'report']
File: /content/submission.csv

First prediction:
FINDINGS:
LIVER: Normal in size and attenuation. No focal hepatic lesion.

GALLBLADDER AND BILIARY TREE: The gallbladder is surgically absent, with surgical clips seen in the gallbladder fossa. No biliary ductal dilatation.

PANCREAS: Normal in size and contour.

SPLEEN: Small splenic calcification is present.

ADRENAL GLANDS: Normal in size and morphology.

KIDNEYS AND URETERS: Normal appearance without renal calculi or hydronephrosis.

URINARY BLADDER: The urinary bladder is surgically absent.

REPRODUCTIVE: The appendix is surgically absent, with surgical suture at the cecum. The remaining reproductive organs are normal in appearance.

MAJOR VESSELS: The abdominal aorta is normal in caliber.

PERITONEUM: No ascites. No pneumoperitoneum.

ABDOMINAL WALL: The abdominal wall is unremarkable. No hernia.

LYMPH NODES: No significant mesenteric or retroperitoneal lymphadenopathy.

S

In [27]:
df = pd.read_csv('/content/submission.csv')

In [28]:
df

,case_id,report
0,00320399-b3ed-447c-be07-a3e3b9af6e2e,FINDINGS:\nLIVER: Normal in size and attenuati...
1,01b1c9ab-800b-4963-b11d-9d503952e57c,FINDINGS:\nVERTEBRAE: Mild lower lumbar facet ...
2,02aae38a-088b-4cae-bfdc-f0fb38da565d,FINDINGS:\nVERTEBRAE: There are degenerative c...
3,045ea333-6d30-4edf-9d33-8ed3feb0772d,FINDINGS:\nVERTEBRAE: A benign-appearing verte...
4,06ef946f-5b3b-4b7c-a64f-58c184dca4e6,FINDINGS:\nBONES: No acute fracture or disloca...
...,...,...
127,d8c7357a-d804-4a1b-9eb7-4de840baf9f7,FINDINGS:\nBONES: No acute fracture or focal a...
128,ddac2ed8-a4f4-4f37-960a-49b27be1d7e5,FINDINGS:\nBONES: No acute fracture is identif...
129,e9721e29-b1d3-4b38-9369-30cec1e63bc0,FINDINGS:\nBOWEL: There is a nonobstructive bo...
130,f6b3620a-65d6-49f7-9fe8-13a3ed4681fc,FINDINGS:\nUTERUS/CERVIX: The uterus appears n...
